# Detecção de Anomalias em Transações Financeiras

## Objetivo

Este projeto tem como objetivo realizar uma análise exploratória de um conjunto de dados de transações financeiras utilizando Python e Pandas, identificando padrões que diferenciam transações legítimas de transações fraudulentas.

O desenvolvimento seguirá as etapas clássicas de um projeto de Ciência de Dados:

1. Coleta dos dados;
2. Compreensão da estrutura do conjunto de dados;
3. Limpeza e preparação;
4. Análise exploratória;
5. Extração de padrões;
6. Construção de modelos de Machine Learning.

## 1. Importação das bibliotecas

Nesta etapa serão importadas as bibliotecas necessárias para realizar a leitura e a manipulação do conjunto de dados.

A biblioteca principal utilizada será o **Pandas**, responsável por fornecer estruturas de dados eficientes, como o DataFrame.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Carregamento do conjunto de dados

O conjunto de dados utilizado contém informações sobre transações realizadas por cartões de crédito.

Neste projeto, o dataset será carregado diretamente de uma URL pública utilizando a função `read_csv()` da biblioteca Pandas. Essa abordagem elimina a necessidade de armazenar o arquivo no repositório, tornando o projeto mais leve e fácil de reproduzir.

In [3]:
# Dataset público de transações de cartão de crédito
DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(DATASET_URL)

In [ ]:
# Para mostrar as 5 primeiras linhas do dataset e confirmação visual do carregamento do dataset 
df.head()

## 3. Compreensão inicial do conjunto de dados

Antes de iniciar qualquer análise, é importante conhecer a estrutura do DataFrame, verificando sua dimensão, os nomes das colunas, os tipos de dados e algumas estatísticas descritivas.

In [ ]:
# Retorna uma tupla com o número de linhas e colunas do dataset
df.shape

In [ ]:
# Lista o nome de todas as colunas
# Ajuda a conhecer quais atributos estão disponíveis para análise
df.columns

In [ ]:
# Função que informa quantidade de linhas, colunas, tipos de dados e quantidade de valores nulos por coluna
# Também mostra consumo de memória do dataset
df.info()

In [ ]:
# Calcula automaticamente medidas estatísticas para todas as colunas numéricas do dataset
df.describe()

## 4. Verificação da qualidade dos dados

Nesta etapa será verificada a existência de valores ausentes (nulos), uma das primeiras verificações realizadas durante o processo de limpeza de dados.

In [ ]:
# A função isnull() gera uma tabela contendo True para valores nulos e False para valores não nulos 
# A função sum() soma os valores True, retornando a quantidade de valores nulos por coluna
df.isnull().sum()

## 5. Análise da variável alvo

A coluna `Class` identifica se uma transação é legítima ou fraudulenta.

- **0** representa uma transação normal.
- **1** representa uma transação fraudulenta.

Antes de construir qualquer modelo, é importante verificar como essas classes estão distribuídas.

In [ ]:
# Método value_counts() contabiliza quantas vezes cada valor aparece na coluna
# Para este conjunto de dados, espera-se encontrar um número muito maior de transações normais do que fradulentas
df["Class"].value_counts()

## 6. Preparação dos dados para modelagem

Para treinar um modelo de Machine Learning, o conjunto de dados será dividido em:

- **Variáveis preditoras (`X`)**: informações utilizadas pelo modelo para reconhecer padrões;
- **Variável alvo (`y`)**: resultado que o modelo deverá prever.

Neste projeto, a coluna `Class` será utilizada como variável alvo:

- `0`: transação legítima;
- `1`: transação fraudulenta.

In [ ]:
# Variáveis utilizadas pelo modelo para realizar as previsões
X = df.drop(columns="Class")

# Variável alvo que o modelo tentará prever
y = df["Class"]

In [ ]:
# Confirmação das dimensões das variáveis X e y
print(f"Dimensões de X: {X.shape}")
print(f"Dimensões de y: {y.shape}")

### Divisão entre treino e teste

Os dados serão divididos em dois grupos:

- **70% para treinamento**: utilizados para o modelo aprender os padrões;
- **30% para teste**: utilizados para avaliar o modelo em transações que ele não viu durante o treinamento.

A divisão estratificada preserva, nos dois grupos, aproximadamente a mesma proporção de transações legítimas e fraudulentas existente no conjunto original.

In [ ]:
#test_size: 30% do dataset será utilizado para teste
# random_state: garante que a divisão do dataset seja sempre a mesma
#stratify: garante que a proporção entre fraudes e não fraudes seja preservada
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [ ]:
print(f"Dados de treino: {X_train.shape[0]} transações")
print(f"Dados de teste: {X_test.shape[0]} transações")

print("\nDistribuição percentual no treino:")
print(y_train.value_counts(normalize=True).mul(100))

print("\nDistribuição percentual no teste:")
print(y_test.value_counts(normalize=True).mul(100))

## 7. Treinamento do modelo de Regressão Logística

A Regressão Logística é um modelo de classificação utilizado para estimar a probabilidade de uma observação pertencer a determinada classe.

Neste projeto, ela será utilizada como **modelo de referência (baseline)**. Seu desempenho servirá como ponto de comparação para técnicas e modelos posteriores.

Antes do treinamento, as variáveis serão padronizadas com `StandardScaler`, evitando que atributos com escalas muito diferentes prejudiquem o processo de otimização do modelo.

In [ ]:
# A Pipeline agrupa dois passos: a padronização dos dados e o treinamento do modelo
# O StandardScaler padroniza os dados para que tenham média 0 e desvio padrão
# O LogisticRegression é o modelo de regressão logística que será treinado com os dados padronizados
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [ ]:
# Treinamento do modelo com os dados de treino
logistic_model.fit(X_train, y_train)

In [ ]:
# Previsões para transações que o modelo não viu durante o treinamento
y_pred = logistic_model.predict(X_test)

In [ ]:
# Visualização das primeiras previsões e dos respectivos valores reais
prediction_comparison = pd.DataFrame(
    {
        "Classe real": y_test.iloc[:10].values,
        "Classe prevista": y_pred[:10]
    }
)

prediction_comparison

## 8. Avaliação do modelo

Como o conjunto de dados é altamente desbalanceado, a acurácia isoladamente pode produzir uma impressão enganosa sobre a qualidade do modelo.

As principais métricas analisadas serão:

- **Precision**: entre as transações classificadas como fraude, quantas realmente eram fraudulentas;
- **Recall**: entre todas as fraudes reais, quantas o modelo conseguiu identificar;
- **F1-score**: média harmônica entre precision e recall;
- **Accuracy**: proporção total de previsões corretas.

Neste problema, será dada atenção especial ao **recall da classe 1**, pois uma fraude não identificada representa uma transação fraudulenta que passou despercebida.

In [ ]:
# Avaliação do modelo com métricas de classificação
# Valores exatos podem diferir um pouco dos apresentados no vídeo do bootcamp
# Foi feita divisão própria dos dados, padronização e e imports de bibliotecas que podem produzir pequenas diferenças
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Transação normal", "Fraude"],
        digits=4
    )
)

In [ ]:
# Matriz de confusão
# Produzirá uma tabela 2x2 mostrando a quantidade de acertos e erros do modelo para cada classe 
# O campo que pode levanat preocupação é o "Previsto: normal" na linha "Real: fraude", que indica a quantidade de fraudes que o modelo classificou erroneamente como transações normais
from sklearn.metrics import confusion_matrix

confusion_matrix_result = confusion_matrix(y_test, y_pred)

confusion_matrix_df = pd.DataFrame(
    confusion_matrix_result,
    index=["Real: normal", "Real: fraude"],
    columns=["Previsto: normal", "Previsto: fraude"]
)

confusion_matrix_df

## 9. Curva ROC e métrica ROC-AUC

A curva ROC avalia o comportamento do modelo em diferentes limiares de classificação.

- O eixo horizontal representa a taxa de falsos positivos;
- O eixo vertical representa a taxa de verdadeiros positivos, equivalente ao recall;
- Uma curva próxima do canto superior esquerdo indica melhor desempenho;
- Uma linha diagonal representa um classificador semelhante a uma decisão aleatória.

A métrica ROC-AUC resume a área abaixo dessa curva. Valores próximos de `1` indicam maior capacidade de separação entre as classes, enquanto valores próximos de `0.5` indicam um comportamento próximo do aleatório.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Probabilidade prevista de cada transação pertencer à classe de fraude
y_probability = logistic_model.predict_proba(X_test)[:, 1]

false_positive_rate, true_positive_rate, roc_thresholds = roc_curve(
    y_test,
    y_probability
)

roc_auc = roc_auc_score(y_test, y_probability)

print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
# Gráfico da curva ROC
plt.figure(figsize=(8, 6))

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"Regressão Logística (AUC = {roc_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Classificador aleatório"
)

plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos (Recall)")
plt.title("Curva ROC")
plt.legend()
plt.grid()
plt.show()

## 10. Curva Precision-Recall

A curva Precision-Recall é particularmente relevante em conjuntos de dados desbalanceados.

Ela mostra o equilíbrio entre:

- **Recall**: proporção das fraudes reais que foram identificadas;
- **Precision**: proporção dos alertas de fraude que estavam corretos.

Em geral, aumentar o recall faz com que o modelo identifique mais fraudes, mas também pode aumentar os falsos positivos e reduzir a precision. Esse equilíbrio é chamado de **trade-off**.